# 01: Station selection (B.2)

Goal: choose ~20 stations for the project. Uses the **training period only** (2024-01 .. 2025-06),
so the validation and test periods don't influence this design decision.

Data: Rijden de Treinen (CC BY 4.0). Needs `make download` and `data/raw/stations-2023-09-nl.csv`.

## Setup
A notebook's working directory is the folder it lives in (`notebooks/`), not the project root.
So we find the project root first and build all paths from it.

In [ ]:
import time
from pathlib import Path

import duckdb

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
STATIONS_CSV = RAW / "stations-2023-09-nl.csv"

# Training period only: zero-padded names sort by date, so a string comparison works.
train_files = sorted(str(p) for p in RAW.glob("services-*.csv.gz") if p.name < "services-2025-07")
print(f"{len(train_files)} files: {Path(train_files[0]).name} .. {Path(train_files[-1]).name}")

con = duckdb.connect()  # in-memory database, lives as long as the kernel

## Q0: columns and types DuckDB infers (one file)

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_csv('{train_files[0]}')").show()

## Q1: stations file, by country and type

In [ ]:
con.sql(f"CREATE OR REPLACE TABLE stations AS SELECT * FROM read_csv('{STATIONS_CSV}')")
con.sql(
    "SELECT country, type, count(*) AS n FROM stations GROUP BY ALL ORDER BY country, n DESC"
).show()

## Load departures (slow step, run once)
Decompress the 18 files once and keep only the columns we need. Every later cell queries this
in-memory table, so it's fast. The last stop of a trip has no departure, so we filter those out.

In [ ]:
t = time.time()
con.sql(f"""
    CREATE OR REPLACE TABLE deps AS
    SELECT
        "Stop:Station code"        AS code,
        "Service:Date"             AS service_date,
        "Service:Type"             AS service_type,
        "Service:Company"          AS company,
        "Stop:Departure delay"     AS delay,
        "Stop:Departure cancelled" AS cancelled
    FROM read_csv({train_files})
    WHERE "Stop:Departure time" IS NOT NULL
""")
n = con.sql("SELECT count(*) FROM deps").fetchall()[0][0]
print(f"loaded {n:,} departures in {time.time() - t:.0f}s")

## Q2: per station
- `LEFT JOIN` keeps codes that are missing from the stations file (they get `NULL` type).
- `avg((delay >= 5)::INT)` = share late; `FILTER (WHERE NOT cancelled)` = among trains that ran.

**Known flaw:** `deps_per_day` divides by the days a station *appears*, not the days in the period.
That's misleading for rare codes (see check 1 below).

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE per_station AS
    SELECT
        d.code,
        s.name_long                                                        AS name,
        s.type,
        round(count(*) / count(DISTINCT d.service_date))                   AS deps_per_day,
        round(100 * avg((d.delay >= 5)::INT) FILTER (WHERE NOT d.cancelled), 1) AS pct_late5,
        round(100 * avg(d.cancelled::INT), 1)                              AS pct_cancelled,
        string_agg(DISTINCT d.company, ',')                                AS companies
    FROM deps d
    LEFT JOIN stations s ON s.code = d.code
    GROUP BY ALL
""")
con.sql("SELECT * FROM per_station ORDER BY deps_per_day DESC LIMIT 45").show(
    max_rows=45, max_width=200
)

## Q3: by station type

In [ ]:
con.sql("""
    SELECT type, count(*) AS stations, median(deps_per_day) AS median_deps_day,
           round(median(pct_late5), 1) AS median_pct_late5, max(pct_late5) AS max_pct_late5
    FROM per_station GROUP BY type ORDER BY median_deps_day DESC
""").show()

## Q4: codes in the archive that are missing from the stations file

In [ ]:
con.sql("""
    SELECT code, deps_per_day FROM per_station WHERE type IS NULL
    ORDER BY deps_per_day DESC LIMIT 10
""").show()

## Q5: service types overall
`sum(count(*)) OVER ()` is a window function: the total over all groups, without merging them,
so `pct` is each type's share of all departures.

In [ ]:
con.sql("""
    SELECT service_type, count(*) AS n,
           round(100 * count(*) / sum(count(*)) OVER (), 2)                     AS pct,
           round(100 * avg((delay >= 5)::INT) FILTER (WHERE NOT cancelled), 1) AS pct_late5
    FROM deps GROUP BY service_type ORDER BY n DESC
""").show(max_rows=30)

## Q6: service types at the suspicious codes

In [ ]:
con.sql("""
    SELECT code, service_type, count(*) AS n FROM deps
    WHERE code IN ('ASDN', 'BEURS', 'VMW', 'CPBR')
    GROUP BY ALL ORDER BY code, n DESC
""").show()

## Overall late ≥5 rate

In [ ]:
overall = con.sql(
    "SELECT round(100 * avg((delay >= 5)::INT) FILTER (WHERE NOT cancelled), 1) FROM deps"
).fetchall()[0][0]
print(f"overall late>=5 rate: {overall} %")

---
# Your checks
Write your own queries in the empty cells below.

**Check 1: fix `deps_per_day`.** Divide by the number of days in the whole period, not the days the station appears.
What happens to ASDN? Do the big stations change?

In [ ]:
# Fix: divide by ALL days in the period (scalar subquery), not by the days the station appears.
con.sql("""
    CREATE OR REPLACE TABLE per_station_fixed AS
    SELECT
        d.code,
        s.name_long                                                       AS name,
        s.type,
        count(*)                                                          AS departures,
        count(DISTINCT d.service_date)                                    AS days_present,
        round(count(*) / (SELECT count(DISTINCT service_date) FROM deps)) AS deps_per_day,
        round(count(*) / count(DISTINCT d.service_date))                  AS old_deps_per_day,
        round(100 * avg((d.delay >= 5)::INT) FILTER (WHERE NOT d.cancelled), 1) AS pct_late5
    FROM deps d
    LEFT JOIN stations s ON s.code = d.code
    GROUP BY ALL
""")

n_days = con.sql("SELECT count(DISTINCT service_date) FROM deps").fetchall()[0][0]
print("Days in the period:", n_days)

print("The suspicious codes, old vs fixed:")
con.sql("""
    SELECT code, departures, days_present, old_deps_per_day, deps_per_day
    FROM per_station_fixed
    WHERE code IN ('ASDN', 'BEURS', 'VMW', 'CPBR', 'SLG', 'HRNT')
    ORDER BY departures DESC
""").show()

print("New top 30:")
con.sql("SELECT * FROM per_station_fixed ORDER BY deps_per_day DESC LIMIT 30").show(
    max_rows=30, max_width=200
)

print("Dutch stations above each departures/day threshold:")
con.sql("""
    SELECT count(*) FILTER (WHERE deps_per_day >= 100) AS ge100,
           count(*) FILTER (WHERE deps_per_day >= 200) AS ge200,
           count(*) FILTER (WHERE deps_per_day >= 300) AS ge300
    FROM per_station_fixed
    WHERE type IS NOT NULL
""").show()

**Check 2: remove replacement services.** Filter out `service_type LIKE '%ipv trein'`.
How do the overall rate and the top-station list change?

In [ ]:
# Keep `deps` as is (raw) and build a filtered table next to it, so before/after can be compared.
con.sql("""
    CREATE OR REPLACE TABLE trains AS
    SELECT * FROM deps
    WHERE service_type NOT LIKE '%ipv trein'   -- replacement bus/metro/taxi: delay not measured
""")

LATE5 = "round(100 * avg((delay >= 5)::INT) FILTER (WHERE NOT cancelled), 2)"
CANC = "round(100 * avg(cancelled::INT), 2)"
INTL = """('Eurostar', 'ICE International', 'EuroCity', 'Eurocity Direct', 'Nightjet',
           'European Sleeper', 'Nachttrein', 'Int. Trein')"""

con.sql(f"""
    SELECT 'all rows' AS subset, count(*) AS n, {LATE5} AS late5, {CANC} AS canc
    FROM deps
    UNION ALL
    SELECT 'trains only', count(*), {LATE5}, {CANC}
    FROM trains
    UNION ALL
    SELECT 'trains, no international', count(*), {LATE5}, {CANC}
    FROM trains WHERE service_type NOT IN {INTL}
""").show()

# Per-station table on the filtered data, with the fixed per-day metric and extra context columns.
con.sql(f"""
    CREATE OR REPLACE TABLE ps AS
    SELECT
        d.code, s.name_long AS name, s.type,
        round(s.geo_lat, 2) AS lat, round(s.geo_lng, 2) AS lng,
        round(count(*) / (SELECT count(DISTINCT service_date) FROM trains)) AS deps_day,
        round(100 * avg((d.delay >= 5)::INT) FILTER (WHERE NOT d.cancelled), 1) AS late5,
        round(100 * avg(d.cancelled::INT), 1) AS canc,
        round(100 * avg((d.service_type IN ('Intercity', 'Intercity direct', 'Sneltrein'))::INT))
            AS pct_ic,
        round(100 * avg((d.service_type IN {INTL})::INT), 1) AS pct_intl
    FROM trains d
    LEFT JOIN stations s ON s.code = d.code
    GROUP BY ALL
""")
con.sql("SELECT * FROM ps ORDER BY deps_day DESC LIMIT 10").show(max_width=200)

**Check 3: robustness.** Are the per-station late rates similar in 2024 alone vs H1 2025?
Stations whose rate changes a lot are worth noting.

In [ ]:
# Per-station late rate in each half-year. Compare H1 with H1: H2 contains autumn (leaf fall).
con.sql("""
    CREATE OR REPLACE TABLE stab AS
    SELECT
        code,
        round(100 * avg((delay >= 5)::INT)
            FILTER (WHERE NOT cancelled AND service_date <  '2024-07-01'), 1) AS l24h1,
        round(100 * avg((delay >= 5)::INT)
            FILTER (WHERE NOT cancelled AND service_date >= '2024-07-01'
                    AND service_date < '2025-01-01'), 1) AS l24h2,
        round(100 * avg((delay >= 5)::INT)
            FILTER (WHERE NOT cancelled AND service_date >= '2025-01-01'), 1) AS l25h1
    FROM trains
    GROUP BY code
""")

print("Do stations keep their ranking? Correlation across stations with >= 100 departures/day:")
con.sql("""
    SELECT count(*) AS stations,
           round(corr(l24h1, l25h1), 2) AS corr_h1_vs_h1,
           round(median(abs(l25h1 - l24h1)), 2) AS median_abs_change
    FROM stab JOIN ps USING (code)
    WHERE ps.deps_day >= 100 AND ps.type IS NOT NULL
""").show()

print("Network-wide, by month (is there drift? one-off events?):")
con.sql(f"""
    SELECT strftime(service_date, '%Y-%m') AS month, {LATE5} AS late5, {CANC} AS canc
    FROM trains GROUP BY month ORDER BY month
""").show(max_rows=20)

print("Candidates: Dutch stations, >= 200/day, plus (knooppunt)stoptrein stations >= 120/day:")
con.sql("""
    SELECT p.*, s.l24h1, s.l24h2, s.l25h1
    FROM ps p JOIN stab s USING (code)
    WHERE p.type IS NOT NULL
      AND (p.deps_day >= 200 OR (p.type ILIKE '%stoptrein%' AND p.deps_day >= 120))
    ORDER BY p.type, p.deps_day DESC
""").show(max_rows=120, max_width=250)

## My station selection

20 stations, chosen on the **training period only**, after removing replacement services.

| # | code | station | type | late ≥5 | why |
|---|---|---|---|---|---|
| 1 | ASD | Amsterdam Centraal | megastation | 6.1% | agent demo ("my 8:15 to Amsterdam"); 3.5% international |
| 2 | UT | Utrecht Centraal | megastation | 4.3% | busiest hub (1,047/day); the centre of the network |
| 3 | RTD | Rotterdam Centraal | megastation | 9.1% | high delay; most international trains (7.8%) |
| 4 | SHL | Schiphol Airport | megastation | 9.5% | highest-delay hub; biggest change 2024→2025 (11.2→6.4%) |
| 5 | EHV | Eindhoven Centraal | megastation | 5.7% | biggest hub in the south |
| 6 | GVC | Den Haag Centraal | megastation | 2.0% | terminus: trains *start* here, so low departure delay; contrast with RTD |
| 7 | ASDZ | Amsterdam Zuid | intercitystation | 6.1% | Amsterdam's 2nd hub; the agent's backup route for ASD |
| 8 | AMF | Amersfoort Centraal | knooppuntIC | 5.9% | central junction towards the east/north |
| 9 | LEDN | Leiden Centraal | knooppuntIC | 4.8% | Randstad west corridor (ASD–SHL–LEDN–GV) |
| 10 | ZL | Zwolle | knooppuntIC | 3.9% | gateway to the north-east |
| 11 | GN | Groningen | knooppuntIC | 1.3% | far north; lowest late rate; regional (Arriva) network |
| 12 | AH | Arnhem Centraal | knooppuntIC | 3.7% | east; ICE to Germany |
| 13 | HGL | Hengelo | knooppuntIC | 5.7% | far east (Twente); stable across years |
| 14 | BD | Breda | knooppuntIC | 10.6% | highest late rate of the hubs; south-west |
| 15 | HT | 's-Hertogenbosch | knooppuntIC | 7.3% | southern junction; high delay and cancellations |
| 16 | MT | Maastricht | knooppuntIC | 2.6% | far south, terminus; low delay but high cancellations (10.8%) |
| 17 | DV | Deventer | knooppuntIC | 7.6% | 74% intercity; high delay that's stable across years |
| 18 | WD | Woerden | knooppuntStoptrein | 4.1% | ~99% sprinters: a pure local-train hub |
| 19 | GDM | Geldermalsen | knooppuntStoptrein | 5.5% | rural junction; sprinter-only; big 2024→2025 change |
| 20 | SDM | Schiedam Centrum | stoptreinstation | 7.2% | busy non-hub station; high delay |

**Coverage:** 16% of all training departures (~8,500/day), late ≥5 = 5.8%, cancelled = 6.8%.
The model is scoped to these stations; it is not expected to generalise to small stations.
